# WordEmbedding 기반 텍스트 분석 연습
> [02_text_analysis.ipynb]의 develop

1. 적절한 데이터셋을 찾거나 생성하고, 적절한 전처리를 진행한다. [01_preprocessing.ipynb]
2. Word Embedding model을 이용하여 벡터화한다.
3. 입력된 문자열의 긍/부정을 판단한다. (유사도 활용)

In [52]:
import pandas as pd
treasure_df= pd.read_csv('../data/heritage_token.csv')
treasure_text_df= pd.read_csv('../data/heritage_sent.csv')
words = treasure_df['word'].astype(str).unique().tolist()
texts = treasure_text_df['sentence'].astype(str).tolist()

In [53]:
words_set = set(words)

sentences = []

# 문장에서 토큰에 해당하는 것만 남겨놓기
for sentence in texts:
    tokens = okt.nouns(sentence)
    
    v_tokens = [t for t in tokens if t in words_set]
    
    if v_tokens:
        sentences.append(v_tokens)
print(sentences[0])

['충청남도', '청양군', '칠갑산', '자리', '장곡사', '대웅전', '철불', '좌상', '나무', '광배', '배경', '사각형', '대좌']


In [54]:
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=2,
    sg=0
)

In [55]:
model.wv.vectors.shape

(8192, 100)

In [56]:
pd.DataFrame(model.wv.vectors, index=model.wv.index_to_key).head(10)

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
모양,-0.395690,0.527334,0.818598,0.656642,-0.883758,-0.473324,0.707979,1.518044,-1.077898,0.366391,...,0.185812,1.655311,-0.454671,0.855436,1.571192,0.305506,1.240007,0.032878,-0.928166,-0.048093
조선,-0.587141,0.319262,0.466450,-0.482414,1.089789,-1.569786,-0.191271,1.476796,-0.121853,-1.282151,...,0.738397,0.238017,0.940853,0.198999,1.098143,1.493495,0.545345,-1.157247,0.320653,-0.001044
부분,-0.358770,0.293517,0.458725,0.127464,-0.373861,-0.348120,0.355059,0.922928,-0.980881,0.128234,...,0.294100,1.177605,-0.483670,0.571328,0.814472,0.541208,0.961596,0.012199,-0.774010,-0.162125
표현,-0.637479,-0.527432,0.052582,-0.087822,0.281209,-0.610074,0.780097,0.989723,-0.555151,-0.407733,...,-0.277970,1.778830,-0.024999,0.919144,0.916580,0.989704,0.692428,-0.126632,-0.086411,0.406933
모습,-0.413968,0.011553,0.481244,0.283078,-0.104350,-0.404393,0.198850,1.224022,-0.623062,0.032250,...,0.095807,1.135890,0.095633,0.876410,1.142383,0.754870,0.551469,-0.049838,-0.267294,0.098637
불상,-0.399014,-0.421103,0.176031,-0.355609,0.413210,-0.651892,0.267667,0.574096,-0.198159,-0.780633,...,-0.161127,1.362099,0.244229,0.605562,0.706268,0.768111,0.249389,-0.253341,-0.020400,0.606486
무늬,-0.182134,-0.239960,0.691208,0.484890,-0.364414,-0.208531,0.949504,0.520875,-1.216769,0.029947,...,0.720391,1.975907,-1.231452,0.043209,1.019914,0.180482,1.810578,0.073152,-0.643018,-0.028096
지붕,0.065865,1.196458,1.407452,1.251621,-0.808928,-0.991774,-0.117027,2.679444,-0.963715,0.200704,...,0.471462,1.136503,0.348246,0.802499,2.241425,0.318529,0.942185,0.094000,-0.680428,-0.490508
조각,0.072237,-0.004227,0.807465,0.507362,-0.095469,-0.184827,0.089822,0.741465,-0.518211,-0.161200,...,0.140864,1.380305,0.010546,0.384869,1.077365,0.077201,0.566926,0.116924,-0.316167,0.291407
자료,-0.326779,0.516300,0.324250,-0.692562,0.390528,-1.408404,0.134959,1.146630,-0.787747,-0.488524,...,0.635125,0.752840,1.545365,-0.437298,0.989564,0.080498,0.149459,-0.691094,0.656871,0.622233


In [57]:
model.wv.save_word2vec_format('ted_en_w2v')

In [58]:
from gensim.models import KeyedVectors

load_model = KeyedVectors.load_word2vec_format('ted_en_w2v')

In [59]:
load_model.most_similar('조선')

[('고려', 0.9722452759742737),
 ('제작', 0.9583708047866821),
 ('전기', 0.958053708076477),
 ('중기', 0.9574450850486755),
 ('추정', 0.956842303276062),
 ('연대', 0.9431061744689941),
 ('이후', 0.9397111535072327),
 ('초기', 0.9396604299545288),
 ('시기', 0.9271615147590637),
 ('초반', 0.923191249370575)]

In [60]:
#!python -m gensim.scripts.word2vec2tensor --input ted_en_w2v --output ted_en_w2v

### 유사도

In [76]:
import numpy as np

okt = Okt()

# 긍정 부정 단어 리스트
pos_words = ['맑음', '공덕', '생동감', '안정', '으뜸'] 
neg_words = ['함부로', '부족', '결함', '파손', '훼손']

input_text = "국가 유산이 훼손되었다."
model = load_model

# 명사 추출
nouns = [n for n in okt.nouns(input_text) if len(n) > 1]
words = [w for w in nouns if w in model]

pos_score = model.n_similarity(words, [w for w in pos_words if w in model])
neg_score = model.n_similarity(words, [w for w in neg_words if w in model])

print(f"\n{input_text}")
print(f"긍정 유사도: {pos_score:.4f}")
print(f"부정 유사도: {neg_score:.4f}")

if pos_score > neg_score:
    print("결과: 긍정")
else:
    print("결과: 부정")


국가 유산이 훼손되었다.
긍정 유사도: 0.8913
부정 유사도: 0.9376
결과: 부정
